In [25]:
import pandas as pd
import numpy as np

fraud = pd.read_csv("../data/raw/Fraud_Data.csv")

In [26]:
fraud["signup_time"] = pd.to_datetime(fraud["signup_time"])
fraud["purchase_time"] = pd.to_datetime(fraud["purchase_time"])

In [27]:
fraud["time_since_signup"] = (
    fraud["purchase_time"] - fraud["signup_time"]
).dt.total_seconds()

In [28]:
fraud["user_transaction_count"] = fraud.groupby("user_id")["user_id"].transform("count")

fraud["device_transaction_count"] = fraud.groupby("device_id")["device_id"].transform("count")

In [30]:
fraud = fraud.sort_values(["user_id", "purchase_time"])

fraud["time_since_prev_txn"] = fraud.groupby("user_id")["purchase_time"].diff().dt.total_seconds()

fraud["time_since_prev_txn"] = fraud["time_since_prev_txn"].fillna(-1)

fraud["rapid_transaction"] = (fraud["time_since_prev_txn"] < 3600).astype(int)

In [31]:
fraud = fraud.drop(
    columns=[
        "signup_time",
        "purchase_time",
        "user_id",
        "device_id"
    ]
)

In [32]:
from sklearn.model_selection import train_test_split

X = fraud.drop("class", axis=1)
y = fraud["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [33]:
cat_cols = ["source", "browser", "sex"]

X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

In [34]:
from sklearn.preprocessing import StandardScaler

num_cols = [
    "purchase_value",
    "age",
    "time_since_signup",
    "user_transaction_count",
    "device_transaction_count",
    "time_since_prev_txn"
]

scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [35]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [36]:
print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_resampled.value_counts())

Before SMOTE:
class
0    109568
1     11321
Name: count, dtype: int64

After SMOTE:
class
0    109568
1    109568
Name: count, dtype: int64


In [37]:
X_train_resampled.to_csv("../data/processed/X_train_resampled.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)